In [ ]:
def extract_windows(data, winsize='10s'):
    X, Y = [], []
    for t, w in data.resample(winsize, origin='start'):

        # Check window has no NaNs and is of correct length
        # 10s @ 100Hz = 1000 ticks
        if w.isna().any().any() or len(w) != 1000:
            continue

        x = w[['x', 'y', 'z']].to_numpy()
        y = w['label'].mode(dropna=False).item()
    
        X.append(x)
        Y.append(y)

    X = np.stack(X)
    Y = np.stack(Y)

    return X, Y

In [ ]:
import scipy.stats as stats
import scipy.signal as signal
from scipy.stats import kurtosis, skew

from numpy import fft


def extract_features(xyz, sample_rate=100):
    ''' Extract commonly used HAR time-series features. xyz is a window of shape (N,3) '''

    feats = {}

    x, y, z = xyz.T
    
    VM=np.sqrt((x**2) + (y**2) + (z**2)) #compute vecor magnitude 
    
    dt=0.01
    n =1000
    #Calc time-domain features
    
    # Mean and std
    feats['mean_x'],feats['std_x'] = np.mean (x), np.std(x) 
    feats['mean_y'],feats['std_y'] = np.mean (y), np.std(y)
    feats['mean_z'],feats['std_z'] = np.mean (z), np.std(z)
    feats['mean_VM'],feats['std_VM'] = np.mean (VM), np.std(VM) 
    
    #Min, Max, quartiles, and median    
    feats['min_x'], feats['q25_x'], feats['med_x'], feats['q75_x'], feats['max_x'] = np.quantile(x, (0, .25, .5, .75, 1))
    feats['min_y'], feats['q25_y'], feats['med_y'], feats['q75_y'], feats['max_y'] = np.quantile(y, (0, .25, .5, .75, 1))
    feats['min_z'], feats['q25_z'], feats['med_z'], feats['q75_z'], feats['max_z'] = np.quantile(z, (0, .25, .5, .75, 1))
    feats['min_VM'], feats['q25_VM'], feats['med_VM'], feats['q75_VM'], feats['max_VM'] = np.quantile(z, (0, .25, .5, .75, 1))
    


    
    with np.errstate(divide='ignore', invalid='ignore'):  # ignore div by 0 warnings
        # xy, xy, zx correlation
        feats['corr_xy'] = np.nan_to_num(np.corrcoef(x,y)[0,1])
        feats['corr_yz'] = np.nan_to_num(np.corrcoef(y,z)[0,1])
        feats['corr_zx'] = np.nan_to_num(np.corrcoef(z,x)[0,1])
        
    
    
    # Spectral features
    x_fhat, y_fhat, z_fhat, VM_fhat = (np.fft.fft(x),
                                       np.fft.fft(y),
                                       np.fft.fft(z),
                                       np.fft.fft(VM))
    
    x_mag, y_mag, z_mag, VM_mag  = ((np.abs(x_fhat) / n),
                                    (np.abs(y_fhat) / n),
                                    (np.abs(z_fhat) / n),
                                    (np.abs(VM_fhat) / n))
    
    
       
        
    freq = (1 / (dt*n)) * np.arange (n)
    L = np.arange(1, np.floor(n/2), dtype ='int') # only half of the spectrum is important
    freq_half = freq [L]
    
    
    x_mag_half = 2 * x_mag[L]
    x_mag_half[0]= x_mag_half[0]/2 # devide by two to correct the DC
    
    y_mag_half = 2 * y_mag[L]
    y_mag_half[0]= y_mag_half[0]/2 # devide by two to correct the DC
    
    z_mag_half = 2 * z_mag[L]
    z_mag_half[0]= z_mag_half[0]/2 # devide by two to correct the DC
    
    VM_mag_half = 2 * VM_mag[L]
    VM_mag_half[0]= VM_mag_half[0]/2 # devide by two to correct the DC

    
    fft_range = np.linspace(1, 15, 15) # Create a vector from 0-14 for getting the frequencies at those Hz
    fft_range_index = np.in1d(freq, fft_range).nonzero()[0] # Get the indeces
    
    # get power of fft from 0-14
    feats['fft1_x'], feats['fft2_x'], feats['fft3_x'], feats['fft4_x'], feats['fft5_x'],\
    feats['fft6_x'], feats['fft7_x'], feats['fft8_x'], feats['fft9_x'], feats['fft10_x'],\
    feats['fft11_x'], feats['fft12_x'], feats['fft13_x'], feats['fft14_x'], feats['fft15_x'] = x_mag_half[fft_range_index]
    
    feats['fft1_y'], feats['fft2_y'], feats['fft3_y'], feats['fft4_y'], feats['fft5_y'],\
    feats['fft6_y'], feats['fft7_y'], feats['fft8_y'], feats['fft9_y'], feats['fft10_y'],\
    feats['fft11_y'], feats['fft12_y'], feats['fft13_y'], feats['fft14_y'], feats['fft15_y'] = y_mag_half[fft_range_index]

    feats['fft1_z'], feats['fft2_z'], feats['fft3_z'], feats['fft4_z'], feats['fft5_z'],\
    feats['fft6_z'], feats['fft7_z'], feats['fft8_z'], feats['fft9_z'], feats['fft10_z'],\
    feats['fft11_z'], feats['fft12_z'], feats['fft13_z'], feats['fft14_z'], feats['fft15_z'] = z_mag_half[fft_range_index]
    
    feats['fft1_VM'], feats['fft2_VM'], feats['fft3_VM'], feats['fft4_VM'], feats['fft5_VM'],\
    feats['fft6_VM'], feats['fft7_VM'], feats['fft8_VM'], feats['fft9_VM'], feats['fft10_VM'],\
    feats['fft11_VM'], feats['fft12_VM'], feats['fft13_VM'], feats['fft14_VM'], feats['fft15_VM'] = VM_mag_half[fft_range_index]

    # Features from ZHANG et. all MSSE 2011 ** ONLY FROM VM***
    
    VM_mag_half_sorted = list(set(VM_mag_half)) # remove duplicated (if any)
    VM_mag_half_sorted.sort()
    largest_mag_ind= np.in1d(VM_mag_half, VM_mag_half_sorted[-1]).nonzero()[0] # Get the indeces for largest magnitude
    largest2_mag_ind= np.in1d(VM_mag_half, VM_mag_half_sorted[-2]).nonzero()[0] # Get the indeces for 2nd largest magnitude
    feats['p1_VM'],feats['f1_VM']= (VM_mag_half[largest_mag_ind][0], freq_half[largest_mag_ind][0])
    feats['p2_VM'],feats['f2_VM']= (VM_mag_half[largest2_mag_ind][0], freq_half[largest2_mag_ind][0])
    

    
    # Total power for the frequencies between 0.3 and 15 Hz
    feats['totpower_VM'] = (np.sum (VM_mag_half[(np.where(np.logical_and(freq_half>=0.3, freq_half<=15))[0])]))
    
    # Dominant frequency and magnitude between 0.6 and 2.5 Hz
    f6_ind = np.where(freq_half >= 0.6)[0]
    f25_ind = np.where(freq_half <= 2.5)[0]
    freq625 = freq_half[f6_ind[0]:f25_ind[-1]+1]
    VM_mag625 = VM_mag_half[f6_ind[0]:f25_ind[-1]+1]
    largest_mag_ind = np.where(VM_mag625 == np.amax(VM_mag625))[0]  # Find the max

    if largest_mag_ind.size > 0:
        feats['f625_VM'] = freq625[largest_mag_ind[0]]
        feats['p625_VM'] = VM_mag625[largest_mag_ind[0]]


    # Angular features
    
    roll = np.arctan2(y, z)
    pitch = np.arctan2(x, z)
    yaw = np.arctan2(y, x)

    feats['avgroll'] = np.mean(roll)
    feats['avgpitch'] = np.mean(pitch)
    feats['avgyaw'] = np.mean(yaw)
    feats['sdroll'] = np.std(roll)
    feats['sdpitch'] = np.std(pitch)
    feats['sdyaw'] = np.std(yaw)

    return feats

